# Local PySpark ETL Walkthrough: NinjaTrader Trade Log

**Goal:** learn the Spark DataFrame API end-to-end (Extract -> Transform -> Load -> Analyze) by
building a small ETL pipeline against a real CSV export from NinjaTrader, then reproduce the
"Cumulative Net Profit" and "Cumulative Max Drawdown" charts and summary stats from the NinjaTrader
Strategy Analyzer, using Spark instead of NinjaTrader's own engine.

This runs 100% locally in `local[*]` mode — no Databricks workspace or Azure account needed. Every
concept here maps directly onto how you'd do the same thing on Databricks; each section calls that
mapping out explicitly, since the point of this notebook is interview prep as much as it is the ETL
itself.

**Prereqs (run once in your terminal):**

```bash
pip install pyspark pandas matplotlib jupyter
```

**Before running:** put `trades.csv` in the same folder as this notebook (it's included alongside it).


## 1. SparkSession — your entry point

Every Spark program starts by creating (or fetching) a `SparkSession`. It's the object that gives
you a connection to the Spark execution engine — think of it as roughly analogous to a DB
connection, except instead of talking to one database, it's coordinating a cluster of workers (or,
here, threads on your own machine) that will execute your DataFrame transformations in parallel.

**Databricks mapping:** on Databricks, you never write this line yourself — a `spark` session is
already injected into every notebook, pre-attached to whatever cluster you've selected in the UI.
`master("local[*]")` below is the one line that's purely local-only; on Databricks that's replaced
by the cluster's actual driver/worker topology (which you configure in the Compute tab, not in
code).


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, StringType
)
import pandas as pd
import matplotlib.pyplot as plt

spark = (
    SparkSession.builder
    .appName("NinjaTraderETL")
    .master("local[*]")   # use all cores on this machine as "workers"
    .getOrCreate()
)

spark


## 2. Extract — read the raw CSV with an explicit schema

NinjaTrader exports currency columns as formatted strings (`$850.00`, `($850.00)` for negatives),
not numbers, and timestamps as `M/d/yyyy h:mm:ss a` strings. If we let Spark infer the schema, it'll
guess everything is a string and we lose type safety immediately, so instead we declare a
`StructType` up front — this is exactly the discipline production Spark jobs use, because schema
inference (`inferSchema=True`) also forces Spark to do a full extra read-through of the file just to
guess types, which is wasteful (and dangerous) at real data volumes.

This raw, as-exported layer is what the medallion architecture (the pattern Databricks pushes hard)
calls the **Bronze layer**: land the data close to its original form, minimal transformation, so you
always have an unmodified source of truth to reprocess from if a downstream transformation turns out
to be wrong.

**Databricks mapping:** on Databricks this `spark.read.csv(...)` would more commonly be
`spark.readStream` (Auto Loader) if trades were arriving continuously, or a scheduled batch job
reading from a landing zone in ADLS/S3. The Bronze table itself would typically be written as a
**Delta table** rather than Parquet — Delta adds ACID transactions and time travel on top of
Parquet, which matters a lot once multiple jobs are writing/reading concurrently.


In [ ]:
raw_schema = StructType([
    StructField("trade_number",       IntegerType(), True),
    StructField("qty",                IntegerType(), True),
    StructField("entry_price",        DoubleType(),  True),
    StructField("exit_price",         DoubleType(),  True),
    StructField("entry_time_raw",     StringType(),  True),
    StructField("exit_time_raw",      StringType(),  True),
    StructField("entry_name",         StringType(),  True),
    StructField("exit_name",          StringType(),  True),
    StructField("profit_raw",         StringType(),  True),
    StructField("cum_net_profit_raw", StringType(),  True),
    StructField("mae_raw",            StringType(),  True),
    StructField("mfe_raw",            StringType(),  True),
    StructField("bars",               IntegerType(), True),
    StructField("_trailing",          StringType(),  True),  # NinjaTrader exports a trailing comma
])

bronze_df = (
    spark.read
    .option("header", True)
    .schema(raw_schema)
    .csv("trades.csv")
)

print(f"Row count: {bronze_df.count()}")
bronze_df.printSchema()
bronze_df.show(5, truncate=False)


## 3. Transform — clean types, parse currency, derive columns

This is the **Silver layer**: typed, cleaned, deduplicated, one-row-per-trade — still fairly close
to raw grain, but now something you'd actually trust other people's queries to run against.

Key transformations:

- **Currency parsing.** `regexp_replace` strips `$` and `,`; a `when/otherwise` checks for the
  parenthesis-negative convention and flips the sign. This is a very common real-world ETL pattern —
  financial exports almost never hand you clean numeric types.
- **Timestamp parsing.** `to_timestamp` with an explicit format string (`M/d/yyyy h:mm:ss a`) turns
  the string into a real `TimestampType`, which is what lets us later compute durations and window
  functions correctly.
- **Derived columns.** `side` (long/short) is pulled out of the `entry_name` field
  (`sqzhisto-long` / `sqzhisto-short`), and `duration_minutes` is computed from the two timestamps.

All of this is expressed as **DataFrame transformations** — `withColumn`, `select`, `when` — which
are *lazy*: nothing actually executes until an action (like `.show()` or `.count()`) is called.
Spark builds a logical plan first, optimizes it (via Catalyst), and only then runs it. This
lazy-evaluation model is one of the first things interviewers probe on, because it's the main
conceptual jump from pandas (which executes eagerly, line by line).


In [ ]:
def parse_currency(colname: str):
    """Convert NinjaTrader-formatted currency strings like '$850.00' / '($850.00)' to a signed double."""
    c = F.col(colname)
    is_negative = c.startswith("(")
    stripped = F.regexp_replace(c, r"[\$,()]", "")
    numeric = stripped.cast("double")
    return F.when(is_negative, -numeric).otherwise(numeric)


TS_FORMAT = "M/d/yyyy h:mm:ss a"

silver_df = (
    bronze_df
    .withColumn("entry_time", F.to_timestamp("entry_time_raw", TS_FORMAT))
    .withColumn("exit_time",  F.to_timestamp("exit_time_raw",  TS_FORMAT))
    .withColumn("profit",     parse_currency("profit_raw"))
    .withColumn("mae",        parse_currency("mae_raw"))
    .withColumn("mfe",        parse_currency("mfe_raw"))
    .withColumn(
        "side",
        F.when(F.col("entry_name").contains("long"), "long")
         .when(F.col("entry_name").contains("short"), "short")
         .otherwise("unknown")
    )
    .withColumn(
        "duration_minutes",
        (F.col("exit_time").cast("long") - F.col("entry_time").cast("long")) / 60.0
    )
    .withColumn("is_win", F.col("profit") > 0)
    .select(
        "trade_number", "qty", "side", "entry_price", "exit_price",
        "entry_time", "exit_time", "duration_minutes", "bars",
        "exit_name", "profit", "mae", "mfe", "is_win",
    )
)

silver_df.printSchema()
silver_df.orderBy("trade_number").show(10, truncate=False)


## 4. Gold layer — window functions for cumulative P&L and drawdown

This is the interesting Spark-specific part. NinjaTrader's CSV *already* includes a
`Cum. net profit` column, but recomputing it ourselves with a **window function** is exactly the
technique you'd reach for on Databricks any time you need a running total, a rank, a
lag/lead comparison, or (as here) a running maximum — all without collapsing rows via `groupBy`,
which would lose row-level detail.

- `Window.orderBy("trade_number").rowsBetween(unboundedPreceding, currentRow)` defines a frame: "all
  rows from the start of the partition up through the current row, in trade order."
- `F.sum("profit").over(w)` → running cumulative net profit.
- `F.max("cum_net_profit").over(w)` → the running *peak* equity seen so far.
- `drawdown = cum_net_profit - running_max` → distance below the most recent peak, which is exactly
  what NinjaTrader's "Cumulative Max Drawdown" chart plots.

**Databricks mapping:** window functions run identically in Databricks SQL and PySpark — this is one
of the more directly transferable skills. The one thing to know for an interview: without a
`PARTITION BY` clause (we don't need one here since it's a single strategy), Spark has to shuffle all
rows to one partition to compute the running window in order, which doesn't scale to huge datasets
without care. At real volume you'd partition by something (e.g. `instrument`, `strategy_id`) so each
partition's window can be computed independently and in parallel.


In [ ]:
trade_order = Window.orderBy("trade_number").rowsBetween(Window.unboundedPreceding, Window.currentRow)

gold_df = (
    silver_df
    .withColumn("cum_net_profit", F.round(F.sum("profit").over(trade_order), 2))
    .withColumn("running_max_equity", F.max("cum_net_profit").over(trade_order))
    .withColumn("drawdown", F.round(F.col("cum_net_profit") - F.col("running_max_equity"), 2))
)

gold_df.select(
    "trade_number", "entry_time", "side", "profit", "cum_net_profit", "drawdown"
).orderBy("trade_number").show(10, truncate=False)

max_drawdown = gold_df.agg(F.min("drawdown")).first()[0]
print(f"Max drawdown (Spark-computed): ${max_drawdown:,.2f}")


## 5. Summary statistics — `groupBy` / `agg`, the Spark equivalent of a SQL `GROUP BY`

This reproduces the "All trades / Long trades / Short trades" columns from the NinjaTrader Strategy
Analyzer summary panel. `groupBy("side").agg(...)` is functionally identical to a SQL
`GROUP BY side` with aggregate expressions in the `SELECT` — and in fact, you could write this exact
logic as Spark SQL instead of the DataFrame API (shown at the end of this section), which is worth
knowing cold for an interview: **the DataFrame API and Spark SQL compile down to the same Catalyst
logical plan**, so there's no performance difference — the choice is purely about which is more
readable for a given task.

We compute: trade count, win rate, gross profit, gross loss, net profit, profit factor
(gross profit / |gross loss|), and average win / average loss.


In [ ]:
summary_by_side = (
    gold_df.groupBy("side")
    .agg(
        F.count("*").alias("num_trades"),
        F.round(F.avg(F.col("is_win").cast("double")) * 100, 2).alias("win_rate_pct"),
        F.round(F.sum(F.when(F.col("profit") > 0, F.col("profit")).otherwise(0.0)), 2).alias("gross_profit"),
        F.round(F.sum(F.when(F.col("profit") < 0, F.col("profit")).otherwise(0.0)), 2).alias("gross_loss"),
        F.round(F.sum("profit"), 2).alias("net_profit"),
        F.round(F.avg(F.when(F.col("profit") > 0, F.col("profit"))), 2).alias("avg_win"),
        F.round(F.avg(F.when(F.col("profit") < 0, F.col("profit"))), 2).alias("avg_loss"),
    )
    .withColumn("profit_factor", F.round(F.col("gross_profit") / F.abs(F.col("gross_loss")), 2))
    .orderBy("side")
)

summary_by_side.show(truncate=False)

# --- overall (all trades) row, for comparison to the "All trades" column in NinjaTrader ---
overall = gold_df.agg(
    F.count("*").alias("num_trades"),
    F.round(F.avg(F.col("is_win").cast("double")) * 100, 2).alias("win_rate_pct"),
    F.round(F.sum("profit"), 2).alias("net_profit"),
)
overall.show(truncate=False)


### The same thing in Spark SQL

Registering a DataFrame as a temp view lets you run SQL directly against it — useful to know because
on Databricks, a huge amount of day-to-day work happens in SQL notebooks/dashboards against Delta
tables registered in Unity Catalog, not the DataFrame API.


In [ ]:
gold_df.createOrReplaceTempView("trades")

spark.sql("""
    SELECT
        side,
        COUNT(*)                                   AS num_trades,
        ROUND(AVG(CASE WHEN is_win THEN 1.0 ELSE 0.0 END) * 100, 2) AS win_rate_pct,
        ROUND(SUM(profit), 2)                      AS net_profit
    FROM trades
    GROUP BY side
    ORDER BY side
""").show(truncate=False)


## 6. Load — write the Gold table out

In a real pipeline this is the step that lands the cleaned, aggregated data somewhere durable and
queryable — a Delta table on Databricks, or a data warehouse. Locally, Parquet is the closest
free equivalent (Delta is literally Parquet files + a JSON transaction log on top, so the on-disk
column format is the same thing you're writing here).

`partitionBy("side")` writes long trades and short trades into separate subfolders — this is the
same partitioning concept used to prune reads at scale (e.g. partitioning a real trades table by
`trade_date` so a query for one day only reads one partition's files instead of the whole table).


In [ ]:
(
    gold_df
    .write
    .mode("overwrite")
    .partitionBy("side")
    .parquet("output/trades_gold")
)

print("Wrote Gold layer to output/trades_gold/ (partitioned by side)")

# sanity check: read it back
spark.read.parquet("output/trades_gold").count()


## 7. Reproduce the NinjaTrader charts

Spark is built for large-scale, distributed transformation — it is *not* a plotting library, and at
real data volumes you'd never plot directly from a Spark DataFrame. The standard pattern (identical
on Databricks) is: do the heavy lifting (filtering, aggregating, windowing) in Spark, then
**`.toPandas()`** the already-small result set and hand it to a normal Python plotting library.

`.toPandas()` pulls the full DataFrame into the driver's memory as a single pandas DataFrame — fine
here (158 rows), dangerous on a multi-billion-row Spark DataFrame. Interviewers sometimes probe this
directly: "how would you plot a 500M row Spark DataFrame?" — answer: you don't, you aggregate/sample
it down in Spark first, *then* convert.


In [ ]:
plot_pdf = (
    gold_df
    .select("trade_number", "entry_time", "cum_net_profit", "drawdown")
    .orderBy("trade_number")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(plot_pdf["entry_time"], plot_pdf["cum_net_profit"], color="#2ca02c", alpha=0.6)
ax.plot(plot_pdf["entry_time"], plot_pdf["cum_net_profit"], color="#2ca02c", linewidth=1)
ax.set_title("Cumulative Net Profit (Spark-computed)")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative profit ($)")
ax.axhline(0, color="white", linewidth=0.5)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(plot_pdf["entry_time"], plot_pdf["drawdown"], color="#d62728", alpha=0.6)
ax.plot(plot_pdf["entry_time"], plot_pdf["drawdown"], color="#d62728", linewidth=1)
ax.set_title("Cumulative Max Drawdown (Spark-computed)")
ax.set_xlabel("Date")
ax.set_ylabel("Drawdown ($)")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"Max drawdown: ${plot_pdf['drawdown'].min():,.2f}   (NinjaTrader reported: ($2,750.00))")


## 8. What's different on real Databricks (talking points for the interview)

Everything above ran in a single local process. Here's what actually changes moving to a real
Databricks/Spark cluster job, which is worth being able to talk through fluently:

| Local (this notebook) | Databricks |
|---|---|
| `master("local[*]")`, one process, threads instead of nodes | A real driver + executor cluster (or serverless SQL warehouse) — you pick node type/count, or let autoscaling handle it |
| Reading a CSV off local disk | `spark.read` from cloud object storage (ADLS Gen2 / S3), often via **Auto Loader** for incremental/streaming ingestion of new files as they land |
| Parquet output | **Delta Lake** — Parquet + a transaction log giving ACID writes, `MERGE INTO` (upserts), schema enforcement/evolution, and time travel (`VERSION AS OF`) |
| No layering, just "output/" | **Medallion architecture**: Bronze (raw) -> Silver (cleaned/typed) -> Gold (aggregated, business-ready) — exactly the 3 layers this notebook walked through, just formalized as named tables |
| Nobody governs access to `output/` | **Unity Catalog** — centralized table/column-level permissions, lineage, and audit logging across workspaces |
| You manually re-run this notebook | **Databricks Workflows/Jobs** — scheduled or triggered orchestration, with retries, alerting, and task dependency graphs (this trades CSV landing could trigger the job automatically) |
| One person's local `pip install pyspark` | A shared **cluster runtime** (Databricks Runtime version) with a pinned Spark version and pre-installed libraries, same for every user of that cluster |
| `.toPandas()` on the whole DataFrame | Same pattern, but you'd only ever call it on an already-aggregated, small result — same caution applies |

The core mental model — DataFrames, lazy transformations vs. eager actions, `groupBy`/`agg`, window
functions, Spark SQL being equivalent to the DataFrame API — is identical in both places. That's the
actual leverage of doing this locally: the muscle memory transfers directly, only the infrastructure
around it changes.
